In [ ]:

from pathlib import Path
import re
import warnings
import math

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import spearmanr, norm
from statsmodels.stats.multitest import multipletests
from IPython.display import display, HTML

warnings.filterwarnings("ignore")




BASE_DIR = Path("data")
SCREEN_DIR = BASE_DIR / "celligner_mss_tmbh_pan_cancer_screen"
OUTPUT_DIR = SCREEN_DIR / "direct_transcriptomic_signature_bootstrap_stable"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GDSC_DIR = BASE_DIR / "GDSC"
CTRP_DIR = BASE_DIR / "CTRP"






DE_FILE_OVERRIDES = {
    ("CRC", "H_vs_L"): "data/DE_results/MSS_TMB_H_vs_MSS_TMB_L.csv",
    ("CRC", "H_vs_MSIH"): "data/DE_results/MSI_H_vs_MSS_TMB_H.csv",
    ("UCEC", "H_vs_L"): "data/DE_results/ucec/MSS_TMB_H_vs_MSS_TMB_L.csv",
    ("UCEC", "H_vs_MSIH"): "data/DE_results/ucec/MSI_H_vs_MSS_TMB_H.csv",
}


STABILITY_FILE_OVERRIDES = {
    "CRC": BASE_DIR / "DE_results/stability_analysis/combined_primary_bootstrap_LOO_stability.csv",
    "UCEC": BASE_DIR / "DE_results/ucec_gene_DE/stability_analysis/combined_primary_bootstrap_LOO_stability.csv",
}


STABILITY_CONTRASTS = {
    "H_vs_L": "MSS_TMB_H_vs_MSS_TMB_L",
    "H_vs_MSIH": "MSI_H_vs_MSS_TMB_H",
}



CELL_LINE_EXPRESSION_FILE = BASE_DIR / "celligner_mss_tmbh_screen/celligner_aligned_expression.pkl"


DEPMAP_MODEL_FILE = BASE_DIR / "Model.csv"


PRISM_RESPONSE_FILE = BASE_DIR / "secondary-screen-dose-response-curve-parameters.csv"
PRISM_TREATMENT_FILE = None




DE_FDR_THRESHOLD = 0.05





TOP_N_PER_DIRECTION = 100

MIN_GENES_PER_DIRECTION = 5




MIN_MODELS_PER_DRUG = 20
MIN_TISSUES_PER_DRUG = 3

print("Output:", OUTPUT_DIR)


In [ ]:

def normalize_colname(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def find_col(df, candidates, required=True):
    exact = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in exact:
            return exact[candidate.lower()]

    normalized = {normalize_colname(c): c for c in df.columns}
    for candidate in candidates:
        key = normalize_colname(candidate)
        if key in normalized:
            return normalized[key]

    if required:
        raise KeyError(
            f"Could not find any of {candidates}. "
            f"Available columns: {df.columns.tolist()}"
        )
    return None


def read_table_auto(path, nrows=None):
    path = Path(path)
    suffix = "".join(path.suffixes).lower()

    if suffix.endswith(".xlsx") or suffix.endswith(".xls"):
        return pd.read_excel(path, nrows=nrows)

    if suffix.endswith(".parquet"):
        df = pd.read_parquet(path)
        return df.head(nrows) if nrows is not None else df

    if suffix.endswith(".pkl") or suffix.endswith(".pickle"):
        df = pd.read_pickle(path)
        return df.head(nrows) if nrows is not None else df

    return pd.read_csv(
        path,
        sep=None,
        engine="python",
        nrows=nrows,
    )


def normalize_gene_name(x):
    x = str(x).strip()

    
    
    
    
    if " (" in x:
        x = x.split(" (", 1)[0]
    if "|" in x:
        x = x.split("|", 1)[0]

    return x.strip().upper()


def normalize_cell_name(x):
    if pd.isna(x):
        return pd.NA

    return re.sub(
        r"[^A-Z0-9]+",
        "",
        str(x).upper().strip(),
    )


DRUG_ALIASES = {
    "azd6244": "selumetinib",
    "azd-6244": "selumetinib",
    "pd-0325901": "pd0325901",
    "pd 0325901": "pd0325901",
    "pd0325901": "pd0325901",
    "mk-1775": "mk1775",
    "azd1775": "mk1775",
    "adavosertib": "mk1775",
}


def normalize_drug_name(x):
    if pd.isna(x):
        return pd.NA

    raw = str(x).lower().strip()
    raw = DRUG_ALIASES.get(raw, raw)
    key = re.sub(r"[^a-z0-9]+", "", raw)
    key = DRUG_ALIASES.get(key, key)

    return key


def bh_fdr(pvalues):
    pvalues = pd.Series(pvalues, dtype=float)
    out = pd.Series(np.nan, index=pvalues.index)

    valid = pvalues.notna()
    if valid.any():
        out.loc[valid] = multipletests(
            pvalues.loc[valid],
            method="fdr_bh",
        )[1]

    return out


def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))


def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return None


In [ ]:

def candidate_de_files():
    roots = [
        BASE_DIR / "DE_results",
        BASE_DIR / "de_results",
        BASE_DIR / "coad",
        BASE_DIR / "read",
        BASE_DIR / "CRC",
        BASE_DIR / "UCEC",
        BASE_DIR / "ucec",
        BASE_DIR,
    ]

    found = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue

        for pattern in ["*.csv", "*.tsv", "*.txt"]:
            for path in root.rglob(pattern):
                if path in seen:
                    continue
                seen.add(path)

                name = path.name.lower()
                if any(
                    token in name
                    for token in [
                        "de",
                        "diff",
                        "mss",
                        "tmb",
                        "msi",
                        "edge",
                    ]
                ):
                    found.append(path)

    return found


def score_de_candidate(path, cancer, contrast):
    name = str(path).lower()
    score = 0

    if cancer == "CRC":
        if any(x in name for x in ["crc", "coad", "read", "coadread"]):
            score += 10
        if "ucec" in name or "uter" in name:
            score -= 10

    elif cancer == "UCEC":
        if any(x in name for x in ["ucec", "uter", "endometr"]):
            score += 10
        if any(x in name for x in ["coad", "read", "crc"]):
            score -= 10

    if "tmb" in name:
        score += 4
    if "mss" in name:
        score += 3

    if contrast == "H_vs_MSIH":
        if "msi" in name:
            score += 8
        else:
            score -= 4

    elif contrast == "H_vs_L":
        if any(x in name for x in ["tmb-l", "tmbl", "tmb_l", "low"]):
            score += 5
        if "msi" in name:
            score -= 5

    
    if any(x in name for x in ["vs", "contrast", "comparison"]):
        score += 2

    return score


all_de_candidates = candidate_de_files()

print(f"DE-like files discovered: {len(all_de_candidates)}")

DE_FILES = {}

for key, override in DE_FILE_OVERRIDES.items():
    cancer, contrast = key

    if override is not None:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(path)
        DE_FILES[key] = path
        continue

    ranked = sorted(
        [
            (score_de_candidate(path, cancer, contrast), path)
            for path in all_de_candidates
        ],
        key=lambda x: (-x[0], len(str(x[1]))),
    )

    ranked = [x for x in ranked if x[0] > 0]

    print(f"\n{cancer} / {contrast} candidates:")
    for score, path in ranked[:8]:
        print(f"  score={score:2d}  {path}")

    if not ranked:
        DE_FILES[key] = None
    else:
        DE_FILES[key] = ranked[0][1]

print("\nAutomatically selected DE files:")
for key, path in DE_FILES.items():
    print(key, "->", path)


In [ ]:


missing_de = [
    key for key, path in DE_FILES.items()
    if path is None
]

if missing_de:
    raise FileNotFoundError(
        "Could not automatically identify DE files for: "
        f"{missing_de}. Set them explicitly in DE_FILE_OVERRIDES "
        "in the configuration cell."
    )


In [ ]:

def load_and_standardize_de(path, cancer, contrast):
    df = read_table_auto(path)

    logfc_col = find_col(
        df,
        ["logFC", "log2FoldChange", "log2FC", "log_fold_change"],
    )

    fdr_col = find_col(
        df,
        ["FDR", "padj", "adj.P.Val", "qvalue", "q_value"],
    )

    gene_col = find_col(
        df,
        [
            "gene",
            "gene_symbol",
            "symbol",
            "Gene",
            "SYMBOL",
            "external_gene_name",
        ],
        required=False,
    )

    if gene_col is None:
        
        first_col = df.columns[0]
        if str(first_col).lower().startswith("unnamed"):
            gene_col = first_col
        else:
            
            if not isinstance(df.index, pd.RangeIndex):
                df = df.reset_index().rename(columns={"index": "__gene__"})
                gene_col = "__gene__"
            else:
                gene_col = first_col

    out = pd.DataFrame({
        "gene": df[gene_col].map(normalize_gene_name),
        "logFC_original": pd.to_numeric(df[logfc_col], errors="coerce"),
        "FDR": pd.to_numeric(df[fdr_col], errors="coerce"),
    })

    out = out.dropna(
        subset=["gene", "logFC_original", "FDR"]
    )

    out = out[
        ~out["gene"].isin(["", "NAN", "NONE"])
    ].copy()

    orientation = 1.0 if contrast == "H_vs_L" else -1.0
    out["oriented_logFC"] = (
        out["logFC_original"] * orientation
    )

    out["source_cancer"] = cancer
    out["contrast"] = contrast
    out["source_file"] = str(path)

    
    out = (
        out.sort_values(
            ["FDR", "oriented_logFC"],
            ascending=[True, False],
        )
        .drop_duplicates("gene")
        .reset_index(drop=True)
    )

    return out


de_tables = {
    key: load_and_standardize_de(path, *key)
    for key, path in DE_FILES.items()
}

for key, df in de_tables.items():
    print(
        key,
        "n genes =", len(df),
        "min FDR =", df["FDR"].min(),
    )
    display(df.head())


In [ ]:
def parse_bool_series(x):
    """Robustly parse TRUE/FALSE, 1/0 and boolean stability flags."""
    if pd.api.types.is_bool_dtype(x):
        return x.fillna(False)

    if pd.api.types.is_numeric_dtype(x):
        return x.fillna(0).astype(float).ne(0)

    return (
        x.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "t", "1", "yes", "y"})
    )


def load_stability_table(path, cancer):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Stability file for {cancer} was not found: {path}. "
            "Update STABILITY_FILE_OVERRIDES in the configuration cell."
        )

    df = read_table_auto(path)

    gene_col = find_col(
        df,
        ["Gene", "gene", "gene_symbol", "symbol", "SYMBOL"],
    )
    contrast_col = find_col(
        df,
        ["Contrast", "contrast", "comparison"],
    )
    robust_col = find_col(
        df,
        [
            "bootstrap_robust_effect",
            "bootstrap robust effect",
            "bootstrap_robust",
            "robust_effect",
        ],
    )

    out = pd.DataFrame({
        "gene": df[gene_col].map(normalize_gene_name),
        "stability_contrast": df[contrast_col].astype(str).str.strip(),
        "bootstrap_robust_effect": parse_bool_series(df[robust_col]),
    })

    out = (
        out.dropna(subset=["gene"])
        .drop_duplicates(["gene", "stability_contrast"])
        .reset_index(drop=True)
    )
    out["source_cancer"] = cancer
    out["stability_source_file"] = str(path)
    return out


stability_tables = {
    cancer: load_stability_table(path, cancer)
    for cancer, path in STABILITY_FILE_OVERRIDES.items()
}

stable_genes = {}

for cancer, stability in stability_tables.items():
    print(f"\n{cancer} stability table: {len(stability)} gene/contrast rows")

    for contrast, stability_contrast in STABILITY_CONTRASTS.items():
        sub = stability[
            stability["stability_contrast"] == stability_contrast
        ].copy()

        if sub.empty:
            available = sorted(stability["stability_contrast"].dropna().unique())
            raise ValueError(
                f"Contrast '{stability_contrast}' was not found in the {cancer} "
                f"stability table. Available contrasts: {available}"
            )

        genes = set(
            sub.loc[sub["bootstrap_robust_effect"], "gene"]
        )
        stable_genes[(cancer, contrast)] = genes

        print(
            f"  {contrast} ({stability_contrast}): "
            f"{len(genes)} bootstrap-stable genes"
        )


In [ ]:
def select_signature_genes(
    de,
    stable_gene_set,
    fdr_threshold=DE_FDR_THRESHOLD,
    top_n=TOP_N_PER_DIRECTION,
):
    """
    Select genes that are significant in the primary DE analysis AND
    bootstrap-stable. Direction/ranking come from the primary DE table.
    """
    sig = de[
        (de["FDR"] <= fdr_threshold)
        & (de["gene"].isin(stable_gene_set))
    ].copy()

    
    sig["bootstrap_robust_effect"] = True

    up = sig[
        sig["oriented_logFC"] > 0
    ].copy()

    down = sig[
        sig["oriented_logFC"] < 0
    ].copy()

    up = up.sort_values(
        ["FDR", "oriented_logFC"],
        ascending=[True, False],
    )

    down = down.assign(
        abs_effect=down["oriented_logFC"].abs()
    ).sort_values(
        ["FDR", "abs_effect"],
        ascending=[True, False],
    )

    if top_n is not None:
        up = up.head(top_n)
        down = down.head(top_n)

    up = up.drop(columns=["abs_effect"], errors="ignore")
    down = down.drop(columns=["abs_effect"], errors="ignore")

    return up, down


signature_definitions = {}
signature_gene_rows = []

for (cancer, contrast), de in de_tables.items():
    stable_gene_set = stable_genes[(cancer, contrast)]

    n_primary_sig = int((de["FDR"] <= DE_FDR_THRESHOLD).sum())
    n_primary_and_stable = int(
        (
            (de["FDR"] <= DE_FDR_THRESHOLD)
            & de["gene"].isin(stable_gene_set)
        ).sum()
    )

    up, down = select_signature_genes(
        de,
        stable_gene_set,
    )

    if len(up) < MIN_GENES_PER_DIRECTION:
        print(
            f"WARNING: {cancer}/{contrast} has only "
            f"{len(up)} stable up genes after filtering."
        )

    if len(down) < MIN_GENES_PER_DIRECTION:
        print(
            f"WARNING: {cancer}/{contrast} has only "
            f"{len(down)} stable down genes after filtering."
        )

    signature_definitions[(cancer, contrast)] = {
        "up": up["gene"].tolist(),
        "down": down["gene"].tolist(),
    }

    for direction, part in [("UP_IN_H", up), ("DOWN_IN_H", down)]:
        tmp = part.copy()
        tmp["direction"] = direction
        tmp["n_primary_FDR_sig"] = n_primary_sig
        tmp["n_primary_FDR_sig_and_bootstrap_stable"] = n_primary_and_stable
        tmp["stability_source_file"] = str(STABILITY_FILE_OVERRIDES[cancer])
        signature_gene_rows.append(tmp)

    print(
        f"{cancer} / {contrast}: "
        f"primary FDR<=0.05={n_primary_sig}; "
        f"primary+bootstrap stable={n_primary_and_stable}; "
        f"signature={len(up)} up + {len(down)} down"
    )


if not signature_gene_rows:
    raise RuntimeError("No bootstrap-stable signature genes were selected.")

signature_genes_table = pd.concat(
    signature_gene_rows,
    ignore_index=True,
)

signature_genes_table.to_csv(
    OUTPUT_DIR / "signature_gene_definitions_bootstrap_stable.csv",
    index=False,
)

display(
    signature_genes_table.groupby(
        ["source_cancer", "contrast", "direction"]
    ).size().rename("n_genes").reset_index()
)


In [ ]:

def score_model_metadata_file(path):
    try:
        sample = read_table_auto(path, nrows=5)
    except Exception:
        return -1

    cols = set(sample.columns)
    score = 0

    if any(c in cols for c in ["ModelID", "DepMap_ID", "DepMapID"]):
        score += 10

    if any(
        c in cols
        for c in [
            "StrippedCellLineName",
            "CellLineName",
            "CCLEName",
            "ModelName",
        ]
    ):
        score += 5

    if any(
        c in cols
        for c in [
            "OncotreeLineage",
            "OncotreePrimaryDisease",
            "PrimaryDisease",
        ]
    ):
        score += 3

    return score


def auto_find_model_file():
    candidates = []

    roots = [
        BASE_DIR,
        Path("combined_paper"),
    ]

    patterns = [
        "**/Model.csv",
        "**/model.csv",
        "**/*Model*.csv",
        "**/*model_metadata*.csv",
    ]

    seen = set()

    for root in roots:
        if not root.exists():
            continue

        for pattern in patterns:
            for path in root.glob(pattern):
                if path in seen:
                    continue
                seen.add(path)

                if str(OUTPUT_DIR) in str(path):
                    continue

                score = score_model_metadata_file(path)

                if score > 0:
                    candidates.append((score, path))

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: (-x[0], len(str(x[1])))
    )

    print("Model metadata candidates:")
    for score, path in candidates[:10]:
        print(f"  score={score:2d}  {path}")

    return candidates[0][1]


if DEPMAP_MODEL_FILE is None:
    DEPMAP_MODEL_FILE = auto_find_model_file()

if DEPMAP_MODEL_FILE is None:
    raise FileNotFoundError(
        "Could not find DepMap Model metadata. "
        "Set DEPMAP_MODEL_FILE manually."
    )

DEPMAP_MODEL_FILE = Path(DEPMAP_MODEL_FILE)

print("\nUsing model metadata:", DEPMAP_MODEL_FILE)

models = read_table_auto(DEPMAP_MODEL_FILE)
print(models.shape)
print(models.columns.tolist())
display(models.head())


In [ ]:

def build_model_crosswalk(models):
    depmap_col = find_col(
        models,
        ["ModelID", "DepMap_ID", "DepMapID"],
    )

    name_cols = [
        c for c in [
            "StrippedCellLineName",
            "CellLineName",
            "CCLEName",
            "ModelName",
        ]
        if c in models.columns
    ]

    tissue_col = next(
        (
            c for c in [
                "OncotreeLineage",
                "OncotreePrimaryDisease",
                "PrimaryDisease",
                "lineage",
                "Lineage",
                "OncotreeCode",
            ]
            if c in models.columns
        ),
        None,
    )

    cosmic_col = find_col(
        models,
        ["COSMICID", "COSMIC_ID", "CosmicID"],
        required=False,
    )

    sanger_col = find_col(
        models,
        ["SangerModelID", "SANGER_MODEL_ID"],
        required=False,
    )

    records = []

    for _, row in models.iterrows():
        depmap = row[depmap_col]

        if pd.isna(depmap):
            continue

        names = []

        for c in name_cols:
            value = row[c]

            if pd.isna(value):
                continue

            names.append(normalize_cell_name(value))

            if c == "CCLEName" and "_" in str(value):
                names.append(
                    normalize_cell_name(
                        str(value).split("_", 1)[0]
                    )
                )

        if not names:
            names = [pd.NA]

        tissue = (
            row[tissue_col]
            if tissue_col is not None
            else pd.NA
        )

        cosmic = pd.NA
        if cosmic_col is not None:
            cosmic = pd.to_numeric(
                pd.Series([row[cosmic_col]]),
                errors="coerce",
            ).astype("Int64").iloc[0]

        sanger = (
            str(row[sanger_col])
            if sanger_col is not None and pd.notna(row[sanger_col])
            else pd.NA
        )

        for name in dict.fromkeys(names):
            records.append({
                "DepMap_ID": str(depmap),
                "name_norm": name,
                "tissue": tissue,
                "COSMIC_ID": cosmic,
                "SANGER_MODEL_ID": sanger,
            })

    return pd.DataFrame(records).drop_duplicates()


model_xwalk = build_model_crosswalk(models)


name_counts = (
    model_xwalk.dropna(subset=["name_norm"])
    .groupby("name_norm")["DepMap_ID"]
    .nunique()
)

valid_names = name_counts[
    name_counts == 1
].index

name_lookup = (
    model_xwalk[
        model_xwalk["name_norm"].isin(valid_names)
    ]
    .drop_duplicates("name_norm")
    .set_index("name_norm")["DepMap_ID"]
    .to_dict()
)

cosmic_tmp = (
    model_xwalk.dropna(subset=["COSMIC_ID"])
    [["COSMIC_ID", "DepMap_ID"]]
    .drop_duplicates()
)

cosmic_counts = (
    cosmic_tmp.groupby("COSMIC_ID")["DepMap_ID"]
    .nunique()
)

cosmic_lookup = (
    cosmic_tmp[
        cosmic_tmp["COSMIC_ID"].isin(
            cosmic_counts[cosmic_counts == 1].index
        )
    ]
    .drop_duplicates("COSMIC_ID")
    .set_index("COSMIC_ID")["DepMap_ID"]
    .to_dict()
)

sanger_tmp = (
    model_xwalk.dropna(subset=["SANGER_MODEL_ID"])
    [["SANGER_MODEL_ID", "DepMap_ID"]]
    .drop_duplicates()
)

sanger_counts = (
    sanger_tmp.groupby("SANGER_MODEL_ID")["DepMap_ID"]
    .nunique()
)

sanger_lookup = (
    sanger_tmp[
        sanger_tmp["SANGER_MODEL_ID"].isin(
            sanger_counts[sanger_counts == 1].index
        )
    ]
    .drop_duplicates("SANGER_MODEL_ID")
    .set_index("SANGER_MODEL_ID")["DepMap_ID"]
    .to_dict()
)

model_tissue = (
    model_xwalk[
        ["DepMap_ID", "tissue"]
    ]
    .dropna()
    .drop_duplicates()
)

tissue_counts = (
    model_tissue.groupby("DepMap_ID")["tissue"]
    .nunique()
)

model_tissue = (
    model_tissue[
        model_tissue["DepMap_ID"].isin(
            tissue_counts[tissue_counts == 1].index
        )
    ]
    .drop_duplicates("DepMap_ID")
)

print("Unique name mappings:", len(name_lookup))
print("Unique COSMIC mappings:", len(cosmic_lookup))
print("Unique Sanger mappings:", len(sanger_lookup))
print("Models with tissue annotation:", model_tissue["DepMap_ID"].nunique())


In [ ]:

expected_signature_genes = set()

for signature in signature_definitions.values():
    expected_signature_genes.update(signature["up"])
    expected_signature_genes.update(signature["down"])

print("Unique expected signature genes:", len(expected_signature_genes))


In [ ]:

def expression_file_candidates():
    roots = [
        BASE_DIR,
        Path("combined_paper"),
    ]

    candidates = []
    seen = set()

    allowed_suffixes = {
        ".csv",
        ".tsv",
        ".txt",
        ".gz",
        ".parquet",
        ".pkl",
        ".pickle",
    }

    for root in roots:
        if not root.exists():
            continue

        for path in root.rglob("*"):
            if not path.is_file():
                continue

            if path in seen:
                continue
            seen.add(path)

            suffixes = [s.lower() for s in path.suffixes]

            if not any(s in allowed_suffixes for s in suffixes):
                continue

            name = path.name.lower()

            if any(
                token in name
                for token in [
                    "celligner",
                    "aligned",
                    "expression",
                    "expr",
                ]
            ):
                if any(
                    bad in name
                    for bad in [
                        "prediction",
                        "drug",
                        "result",
                        "clinical",
                        "signature_gene",
                    ]
                ):
                    continue

                candidates.append(path)

    return candidates


def score_expression_candidate(path, expected_genes):
    score = 0

    name = path.name.lower()

    if "celligner" in name:
        score += 8
    if "aligned" in name:
        score += 6
    if "expression" in name or "expr" in name:
        score += 3

    try:
        sample = read_table_auto(path, nrows=200)
    except Exception:
        return -999

    col_genes = {
        normalize_gene_name(c)
        for c in sample.columns
    }

    col_overlap = len(
        col_genes.intersection(expected_genes)
    )

    first_col_values = set()

    if sample.shape[1] > 0:
        first_col_values = {
            normalize_gene_name(x)
            for x in sample.iloc[:, 0].dropna().astype(str)
        }

    row_overlap = len(
        first_col_values.intersection(expected_genes)
    )

    score += 3 * max(col_overlap, row_overlap)

    
    if max(col_overlap, row_overlap) >= 20:
        score += 20

    return score


if CELL_LINE_EXPRESSION_FILE is None:
    expr_candidates = expression_file_candidates()

    ranked_expr = sorted(
        [
            (
                score_expression_candidate(
                    path,
                    expected_signature_genes,
                ),
                path,
            )
            for path in expr_candidates
        ],
        key=lambda x: (-x[0], len(str(x[1]))),
    )

    print("Expression candidates:")
    for score, path in ranked_expr[:15]:
        print(f"  score={score:3d}  {path}")

    if not ranked_expr or ranked_expr[0][0] <= 0:
        raise FileNotFoundError(
            "Could not identify the Celligner-aligned expression matrix. "
            "Set CELL_LINE_EXPRESSION_FILE manually."
        )

    CELL_LINE_EXPRESSION_FILE = ranked_expr[0][1]

CELL_LINE_EXPRESSION_FILE = Path(
    CELL_LINE_EXPRESSION_FILE
)

print("\nUsing expression:", CELL_LINE_EXPRESSION_FILE)


In [ ]:

def load_expression_matrix(path, expected_genes):
    df = read_table_auto(path)

    
    
    if isinstance(df.index, pd.RangeIndex) and df.shape[1] > 1:
        first = df.columns[0]

        
        numeric_fraction = pd.to_numeric(
            df[first],
            errors="coerce",
        ).notna().mean()

        if numeric_fraction < 0.5:
            df = df.set_index(first)

    normalized_cols = {
        c: normalize_gene_name(c)
        for c in df.columns
    }

    normalized_idx = {
        idx: normalize_gene_name(idx)
        for idx in df.index
    }

    col_overlap = len(
        set(normalized_cols.values()).intersection(
            expected_genes
        )
    )

    idx_overlap = len(
        set(normalized_idx.values()).intersection(
            expected_genes
        )
    )

    print(
        "Gene overlap if genes are columns:",
        col_overlap,
    )
    print(
        "Gene overlap if genes are rows:",
        idx_overlap,
    )

    if idx_overlap > col_overlap:
        df = df.T

    
    df.columns = [
        normalize_gene_name(c)
        for c in df.columns
    ]

    numeric = df.apply(
        pd.to_numeric,
        errors="coerce",
    )

    
    numeric = numeric.T.groupby(level=0).mean().T

    
    mapped_ids = []

    for idx in numeric.index.astype(str):
        ach = re.search(r"(ACH-\d+)", idx)

        if ach:
            mapped_ids.append(ach.group(1))
            continue

        mapped_ids.append(
            name_lookup.get(
                normalize_cell_name(idx),
                pd.NA,
            )
        )

    numeric["__DepMap_ID__"] = mapped_ids

    numeric = numeric.dropna(
        subset=["__DepMap_ID__"]
    )

    numeric = (
        numeric.groupby("__DepMap_ID__")
        .mean(numeric_only=True)
    )

    numeric.index.name = "DepMap_ID"

    return numeric


expr = load_expression_matrix(
    CELL_LINE_EXPRESSION_FILE,
    expected_signature_genes,
)

print("Mapped cell-line expression:", expr.shape)
print("Mapped DepMap models:", expr.shape[0])

available_signature_genes = (
    set(expr.columns)
    .intersection(expected_signature_genes)
)

print(
    "Signature genes available in expression:",
    len(available_signature_genes),
    "/",
    len(expected_signature_genes),
)

display(expr.iloc[:5, :8])


In [ ]:


expr_mean = expr.mean(axis=0)
expr_sd = expr.std(axis=0, ddof=0)

nonconstant = (
    expr_sd.notna()
    & (expr_sd > 0)
)

expr_z = (
    expr.loc[:, nonconstant]
    - expr_mean.loc[nonconstant]
) / expr_sd.loc[nonconstant]

signature_score_rows = []
signature_coverage_rows = []

for (cancer, contrast), definition in signature_definitions.items():
    up_requested = definition["up"]
    down_requested = definition["down"]

    up_used = [
        g for g in up_requested
        if g in expr_z.columns
    ]

    down_used = [
        g for g in down_requested
        if g in expr_z.columns
    ]

    if len(up_used) == 0 or len(down_used) == 0:
        raise ValueError(
            f"No usable genes for {cancer}/{contrast}. "
            f"UP={len(up_used)}, DOWN={len(down_used)}"
        )

    score = (
        expr_z[up_used].mean(axis=1)
        - expr_z[down_used].mean(axis=1)
    )

    tmp = pd.DataFrame({
        "DepMap_ID": score.index,
        "source_cancer": cancer,
        "contrast": contrast,
        "signature_score": score.values,
        "n_up_used": len(up_used),
        "n_down_used": len(down_used),
    })

    signature_score_rows.append(tmp)

    signature_coverage_rows.append({
        "source_cancer": cancer,
        "contrast": contrast,
        "n_up_requested": len(up_requested),
        "n_up_used": len(up_used),
        "up_coverage": (
            len(up_used) / len(up_requested)
            if up_requested else np.nan
        ),
        "n_down_requested": len(down_requested),
        "n_down_used": len(down_used),
        "down_coverage": (
            len(down_used) / len(down_requested)
            if down_requested else np.nan
        ),
    })


signature_scores = pd.concat(
    signature_score_rows,
    ignore_index=True,
)

signature_coverage = pd.DataFrame(
    signature_coverage_rows
)

signature_scores = signature_scores.merge(
    model_tissue,
    on="DepMap_ID",
    how="left",
    validate="many_to_one",
)

display(signature_coverage)

signature_coverage.to_csv(
    OUTPUT_DIR / "signature_gene_coverage.csv",
    index=False,
)

signature_scores.to_csv(
    OUTPUT_DIR / "cell_line_direct_signature_scores.csv",
    index=False,
)

display(
    signature_scores.groupby(
        ["source_cancer", "contrast"]
    )["signature_score"].describe()
)


In [ ]:

GDSC1_FILE = GDSC_DIR / "GDSC1_fitted_dose_response_27Oct23.xlsx"
GDSC2_FILE = GDSC_DIR / "GDSC2_fitted_dose_response_27Oct23.xlsx"
GDSC_COMPOUND_FILE = GDSC_DIR / "screened_compounds_rel_8.5.csv"

gdsc1_raw = pd.read_excel(GDSC1_FILE)
gdsc2_raw = pd.read_excel(GDSC2_FILE)

gdsc_compounds = (
    pd.read_csv(GDSC_COMPOUND_FILE)
    if GDSC_COMPOUND_FILE.exists()
    else pd.DataFrame()
)

print("GDSC1:", gdsc1_raw.shape)
print("GDSC2:", gdsc2_raw.shape)


In [ ]:

def standardize_gdsc(df, dataset_name):
    cell_name_col = find_col(
        df,
        ["CELL_LINE_NAME", "cell_line_name", "model_name", "CELL_LINE"],
    )

    cosmic_col = find_col(
        df,
        ["COSMIC_ID", "cosmic_id", "COSMICID"],
        required=False,
    )

    sanger_col = find_col(
        df,
        ["SANGER_MODEL_ID", "SangerModelID", "model_id"],
        required=False,
    )

    drug_id_col = find_col(
        df,
        ["DRUG_ID", "drug_id", "compound_id"],
    )

    drug_name_col = find_col(
        df,
        ["DRUG_NAME", "drug_name", "compound_name"],
    )

    auc_col = find_col(
        df,
        ["AUC", "auc"],
    )

    out = pd.DataFrame({
        "dataset": dataset_name,
        "gdsc_cell_line": df[cell_name_col].astype(str),
        "drug_id": df[drug_id_col],
        "drug": df[drug_name_col].astype(str),
        "auc": pd.to_numeric(
            df[auc_col],
            errors="coerce",
        ),
    })

    if cosmic_col is not None:
        out["COSMIC_ID"] = pd.to_numeric(
            df[cosmic_col],
            errors="coerce",
        ).astype("Int64")
    else:
        out["COSMIC_ID"] = pd.Series(
            pd.NA,
            index=df.index,
            dtype="Int64",
        )

    if sanger_col is not None:
        out["SANGER_MODEL_ID"] = (
            df[sanger_col].astype("string")
        )
    else:
        out["SANGER_MODEL_ID"] = pd.Series(
            pd.NA,
            index=df.index,
            dtype="string",
        )

    out["name_norm"] = (
        out["gdsc_cell_line"]
        .map(normalize_cell_name)
    )

    
    out["DepMap_ID"] = pd.NA

    mapped = out["COSMIC_ID"].map(cosmic_lookup)
    mask = out["DepMap_ID"].isna() & mapped.notna()
    out.loc[mask, "DepMap_ID"] = mapped[mask]

    mapped = out["SANGER_MODEL_ID"].map(sanger_lookup)
    mask = out["DepMap_ID"].isna() & mapped.notna()
    out.loc[mask, "DepMap_ID"] = mapped[mask]

    mapped = out["name_norm"].map(name_lookup)
    mask = out["DepMap_ID"].isna() & mapped.notna()
    out.loc[mask, "DepMap_ID"] = mapped[mask]

    out = out.dropna(
        subset=["DepMap_ID", "auc", "drug"]
    )

    out = (
        out.groupby(
            [
                "dataset",
                "DepMap_ID",
                "drug_id",
                "drug",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(auc=("auc", "median"))
    )

    out["target"] = pd.NA

    return out


gdsc1 = standardize_gdsc(
    gdsc1_raw,
    "GDSC1",
)

gdsc2 = standardize_gdsc(
    gdsc2_raw,
    "GDSC2",
)

display(
    pd.concat([gdsc1, gdsc2])
    .groupby("dataset")
    .agg(
        n_models=("DepMap_ID", "nunique"),
        n_drugs=("drug_id", "nunique"),
        n_rows=("auc", "size"),
    )
)


In [ ]:

CTRP_CURVES = CTRP_DIR / "v20.data.curves_post_qc.txt"
CTRP_EXPERIMENT = CTRP_DIR / "v20.meta.per_experiment.txt"
CTRP_CELL = CTRP_DIR / "v20.meta.per_cell_line.txt"
CTRP_COMPOUND = CTRP_DIR / "v20.meta.per_compound.txt"

ctrp_curves_raw = read_table_auto(CTRP_CURVES)
ctrp_exp_raw = read_table_auto(CTRP_EXPERIMENT)
ctrp_cell_raw = read_table_auto(CTRP_CELL)
ctrp_cpd_raw = read_table_auto(CTRP_COMPOUND)


In [ ]:

curve_exp_col = find_col(
    ctrp_curves_raw,
    ["experiment_id", "experimentid"],
)

curve_cpd_col = find_col(
    ctrp_curves_raw,
    ["master_cpd_id", "mastercpdid", "compound_id", "cpd_id"],
)

curve_auc_col = find_col(
    ctrp_curves_raw,
    ["area_under_curve", "auc", "curve_auc"],
)

exp_id_col = find_col(
    ctrp_exp_raw,
    ["experiment_id", "experimentid"],
)

exp_ccl_col = find_col(
    ctrp_exp_raw,
    ["master_ccl_id", "mastercclid", "ccl_id"],
)

ccl_id_col = find_col(
    ctrp_cell_raw,
    ["master_ccl_id", "mastercclid", "ccl_id"],
)

ccl_name_col = find_col(
    ctrp_cell_raw,
    [
        "ccl_name",
        "ccle_name",
        "cell_line_name",
        "cell_line",
        "name",
    ],
)

cpd_id_col = find_col(
    ctrp_cpd_raw,
    ["master_cpd_id", "mastercpdid", "cpd_id", "compound_id"],
)

cpd_name_col = find_col(
    ctrp_cpd_raw,
    ["cpd_name", "compound_name", "drug_name", "name"],
)

cpd_target_col = find_col(
    ctrp_cpd_raw,
    [
        "gene_symbol_of_protein_target",
        "gene_symbol",
        "target",
        "targets",
        "protein_target",
    ],
    required=False,
)

curves = pd.DataFrame({
    "experiment_id": ctrp_curves_raw[curve_exp_col],
    "drug_id": ctrp_curves_raw[curve_cpd_col],
    "auc": pd.to_numeric(
        ctrp_curves_raw[curve_auc_col],
        errors="coerce",
    ),
}).dropna()

experiments = (
    pd.DataFrame({
        "experiment_id": ctrp_exp_raw[exp_id_col],
        "master_ccl_id": ctrp_exp_raw[exp_ccl_col],
    })
    .drop_duplicates("experiment_id")
)

cell_lines = (
    pd.DataFrame({
        "master_ccl_id": ctrp_cell_raw[ccl_id_col],
        "cell_name": ctrp_cell_raw[ccl_name_col].astype(str),
    })
    .drop_duplicates("master_ccl_id")
)

cell_lines["name_norm"] = (
    cell_lines["cell_name"]
    .map(normalize_cell_name)
)

cell_lines["DepMap_ID"] = (
    cell_lines["name_norm"]
    .map(name_lookup)
)

compounds = pd.DataFrame({
    "drug_id": ctrp_cpd_raw[cpd_id_col],
    "drug": ctrp_cpd_raw[cpd_name_col].astype(str),
})

compounds["target"] = (
    ctrp_cpd_raw[cpd_target_col]
    if cpd_target_col is not None
    else pd.NA
)

compounds = compounds.drop_duplicates("drug_id")

ctrp = (
    curves
    .merge(
        experiments,
        on="experiment_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        cell_lines[
            ["master_ccl_id", "DepMap_ID"]
        ],
        on="master_ccl_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        compounds,
        on="drug_id",
        how="left",
        validate="many_to_one",
    )
)

ctrp = ctrp.dropna(
    subset=["DepMap_ID", "drug", "auc"]
)

ctrp = (
    ctrp.groupby(
        [
            "DepMap_ID",
            "drug_id",
            "drug",
            "target",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(auc=("auc", "median"))
)

ctrp["dataset"] = "CTRP"

display(
    ctrp.agg(
        n_models=("DepMap_ID", "nunique"),
        n_drugs=("drug_id", "nunique"),
        n_rows=("auc", "size"),
    )
)


In [ ]:

def prism_candidate_files():
    roots = [
        BASE_DIR / "PRISM",
        BASE_DIR / "prism",
        BASE_DIR,
    ]

    found = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue

        for path in root.rglob("*"):
            if not path.is_file():
                continue

            if path in seen:
                continue
            seen.add(path)

            name = path.name.lower()

            if not any(
                token in name
                for token in [
                    "prism",
                    "secondary-screen",
                    "dose-response",
                    "curve-parameter",
                ]
            ):
                continue

            if not any(
                s.lower() in {
                    ".csv",
                    ".tsv",
                    ".txt",
                    ".gz",
                    ".parquet",
                }
                for s in path.suffixes
            ):
                continue

            found.append(path)

    return found


def score_prism_response_candidate(path):
    try:
        sample = read_table_auto(path, nrows=5)
    except Exception:
        return -999

    cols_norm = {
        normalize_colname(c)
        for c in sample.columns
    }

    score = 0

    if any(
        normalize_colname(c) in cols_norm
        for c in [
            "DepMap_ID",
            "depmap_id",
            "ModelID",
        ]
    ):
        score += 10

    if "auc" in cols_norm:
        score += 10

    if any(
        normalize_colname(c) in cols_norm
        for c in [
            "drug",
            "drug_name",
            "name",
            "pert_iname",
            "broad_id",
            "treatment_id",
        ]
    ):
        score += 4

    if "secondary" in path.name.lower():
        score += 2

    return score


if PRISM_RESPONSE_FILE is None:
    prism_candidates = prism_candidate_files()

    ranked_prism = sorted(
        [
            (
                score_prism_response_candidate(path),
                path,
            )
            for path in prism_candidates
        ],
        key=lambda x: (-x[0], len(str(x[1]))),
    )

    print("PRISM response candidates:")
    for score, path in ranked_prism[:15]:
        print(f"  score={score:3d}  {path}")

    if not ranked_prism or ranked_prism[0][0] <= 0:
        raise FileNotFoundError(
            "Could not identify raw PRISM AUC data. "
            "Set PRISM_RESPONSE_FILE manually."
        )

    PRISM_RESPONSE_FILE = ranked_prism[0][1]

PRISM_RESPONSE_FILE = Path(
    PRISM_RESPONSE_FILE
)

print("\nUsing PRISM response:", PRISM_RESPONSE_FILE)

prism_raw = read_table_auto(
    PRISM_RESPONSE_FILE
)

print(prism_raw.shape)
print(prism_raw.columns.tolist())


In [ ]:

def discover_prism_treatment_file(
    response_path,
    response_drug_id_col,
):
    search_roots = [
        response_path.parent,
        BASE_DIR / "PRISM",
        BASE_DIR / "prism",
    ]

    candidates = []

    for root in search_roots:
        if not root.exists():
            continue

        for path in root.glob("*"):
            if not path.is_file():
                continue

            if path == response_path:
                continue

            try:
                sample = read_table_auto(path, nrows=5)
            except Exception:
                continue

            id_col = find_col(
                sample,
                [
                    response_drug_id_col,
                    "broad_id",
                    "treatment_id",
                    "pert_id",
                    "drug_id",
                ],
                required=False,
            )

            name_col = find_col(
                sample,
                [
                    "drug",
                    "drug_name",
                    "name",
                    "pert_iname",
                    "compound_name",
                ],
                required=False,
            )

            if id_col is not None and name_col is not None:
                candidates.append(path)

    return candidates[0] if candidates else None


prism_depmap_col = find_col(
    prism_raw,
    ["DepMap_ID", "depmap_id", "ModelID"],
)

prism_auc_col = find_col(
    prism_raw,
    ["auc", "AUC"],
)

prism_drug_name_col = find_col(
    prism_raw,
    [
        "drug",
        "drug_name",
        "name",
        "pert_iname",
        "compound_name",
    ],
    required=False,
)

prism_drug_id_col = find_col(
    prism_raw,
    [
        "broad_id",
        "treatment_id",
        "pert_id",
        "drug_id",
    ],
    required=(prism_drug_name_col is None),
)

prism = pd.DataFrame({
    "DepMap_ID": prism_raw[prism_depmap_col].astype(str),
    "auc": pd.to_numeric(
        prism_raw[prism_auc_col],
        errors="coerce",
    ),
})

if prism_drug_id_col is not None:
    prism["drug_id"] = (
        prism_raw[prism_drug_id_col]
        .astype(str)
    )
else:
    prism["drug_id"] = (
        prism_raw[prism_drug_name_col]
        .astype(str)
    )

if prism_drug_name_col is not None:
    prism["drug"] = (
        prism_raw[prism_drug_name_col]
        .astype(str)
    )
else:
    prism["drug"] = pd.NA

prism["target"] = pd.NA


if prism["drug"].isna().all():
    if PRISM_TREATMENT_FILE is None:
        PRISM_TREATMENT_FILE = discover_prism_treatment_file(
            PRISM_RESPONSE_FILE,
            prism_drug_id_col,
        )

    if PRISM_TREATMENT_FILE is not None:
        treatment = read_table_auto(
            PRISM_TREATMENT_FILE
        )

        tid = find_col(
            treatment,
            [
                prism_drug_id_col,
                "broad_id",
                "treatment_id",
                "pert_id",
                "drug_id",
            ],
        )

        tname = find_col(
            treatment,
            [
                "drug",
                "drug_name",
                "name",
                "pert_iname",
                "compound_name",
            ],
        )

        ttarget = find_col(
            treatment,
            ["target", "targets", "target_name"],
            required=False,
        )

        treatment_small = pd.DataFrame({
            "drug_id": treatment[tid].astype(str),
            "drug_meta": treatment[tname].astype(str),
        })

        if ttarget is not None:
            treatment_small["target_meta"] = treatment[ttarget]
        else:
            treatment_small["target_meta"] = pd.NA

        treatment_small = treatment_small.drop_duplicates(
            "drug_id"
        )

        prism = prism.merge(
            treatment_small,
            on="drug_id",
            how="left",
            validate="many_to_one",
        )

        prism["drug"] = prism["drug"].fillna(
            prism["drug_meta"]
        )

        prism["target"] = prism["target"].fillna(
            prism["target_meta"]
        )


prism["drug"] = prism["drug"].fillna(
    prism["drug_id"].astype(str)
)

prism = prism.dropna(
    subset=["DepMap_ID", "auc", "drug"]
)

prism = (
    prism.groupby(
        [
            "DepMap_ID",
            "drug_id",
            "drug",
            "target",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(auc=("auc", "median"))
)

prism["dataset"] = "PRISM"

display(
    pd.DataFrame({
        "n_models": [prism["DepMap_ID"].nunique()],
        "n_drugs": [prism["drug_id"].nunique()],
        "n_rows": [len(prism)],
    })
)

display(prism.head())


In [ ]:

response_tables = [
    prism[
        [
            "dataset",
            "DepMap_ID",
            "drug_id",
            "drug",
            "target",
            "auc",
        ]
    ],
    gdsc1[
        [
            "dataset",
            "DepMap_ID",
            "drug_id",
            "drug",
            "target",
            "auc",
        ]
    ],
    gdsc2[
        [
            "dataset",
            "DepMap_ID",
            "drug_id",
            "drug",
            "target",
            "auc",
        ]
    ],
    ctrp[
        [
            "dataset",
            "DepMap_ID",
            "drug_id",
            "drug",
            "target",
            "auc",
        ]
    ],
]

responses = pd.concat(
    response_tables,
    ignore_index=True,
)

responses["drug_key"] = (
    responses["drug"].map(normalize_drug_name)
)

display(
    responses.groupby("dataset").agg(
        n_models=("DepMap_ID", "nunique"),
        n_drugs=("drug_id", "nunique"),
        n_rows=("auc", "size"),
    )
)


In [ ]:



print("Signature-score models:", signature_scores["DepMap_ID"].nunique())
print(
    "Signature-score models with tissue:",
    signature_scores.loc[
        signature_scores["tissue"].notna(),
        "DepMap_ID",
    ].nunique(),
)
print(
    "Unique tissues in signature scores:",
    signature_scores["tissue"].dropna().nunique(),
)

diagnostic_rows = []

for (source_cancer, contrast), sig_df in signature_scores.groupby(
    ["source_cancer", "contrast"],
    sort=False,
):
    sig_small = sig_df[
        ["DepMap_ID", "signature_score", "tissue"]
    ].drop_duplicates("DepMap_ID")

    for dataset, response_one in responses.groupby("dataset", sort=False):
        merged = response_one.merge(
            sig_small,
            on="DepMap_ID",
            how="inner",
            validate="many_to_one",
        )

        complete = merged[
            ["DepMap_ID", "drug_id", "drug", "auc", "signature_score", "tissue"]
        ].dropna()

        eligible_drugs = 0

        for (drug_id, drug), drug_df in complete.groupby(
            ["drug_id", "drug"],
            sort=False,
        ):
            data = drug_df.drop_duplicates("DepMap_ID").copy()

            tissue_n = (
                data.groupby("tissue")["DepMap_ID"]
                .nunique()
            )
            eligible_tissues = tissue_n[
                tissue_n >= 2
            ].index

            data = data[
                data["tissue"].isin(eligible_tissues)
            ]

            if (
                data["DepMap_ID"].nunique() >= MIN_MODELS_PER_DRUG
                and data["tissue"].nunique() >= MIN_TISSUES_PER_DRUG
            ):
                eligible_drugs += 1

        diagnostic_rows.append({
            "dataset": dataset,
            "source_cancer": source_cancer,
            "contrast": contrast,
            "signature_models": sig_small["DepMap_ID"].nunique(),
            "signature_models_with_tissue": sig_small.loc[
                sig_small["tissue"].notna(), "DepMap_ID"
            ].nunique(),
            "response_models": response_one["DepMap_ID"].nunique(),
            "overlap_models": merged["DepMap_ID"].nunique(),
            "complete_overlap_models": complete["DepMap_ID"].nunique(),
            "eligible_drugs_before_variance_check": eligible_drugs,
        })

screen_diagnostics = pd.DataFrame(diagnostic_rows)

display(
    screen_diagnostics.sort_values(
        ["source_cancer", "contrast", "dataset"]
    )
)

screen_diagnostics.to_csv(
    OUTPUT_DIR / "pre_screen_diagnostics.csv",
    index=False,
)

In [ ]:
RESULT_COLUMNS = [
    "dataset",
    "source_cancer",
    "contrast",
    "drug_id",
    "drug",
    "drug_key",
    "target",
    "n_models",
    "n_tissues",
    "n_up_genes",
    "n_down_genes",
    "beta_tissue_adjusted",
    "beta_standardized",
    "standard_error_HC3",
    "pvalue",
    "spearman_rho_within_tissue",
    "spearman_pvalue_within_tissue",
    "median_auc",
    "FDR_within_analysis",
    "FDR_global_all_tests",
    "direction",
]


def run_signature_drug_screen(
    response_df,
    score_df,
    min_models=MIN_MODELS_PER_DRUG,
    min_tissues=MIN_TISSUES_PER_DRUG,
):
    rows = []
    rejected = {
        "too_few_models": 0,
        "too_few_tissues": 0,
        "zero_score_variance": 0,
        "zero_auc_variance": 0,
    }

    datasets = response_df["dataset"].unique()

    for (
        source_cancer,
        contrast,
    ), sig_df in score_df.groupby(
        ["source_cancer", "contrast"],
        sort=False,
    ):
        sig_small = sig_df[
            [
                "DepMap_ID",
                "signature_score",
                "tissue",
                "n_up_used",
                "n_down_used",
            ]
        ].drop_duplicates("DepMap_ID")

        for dataset in datasets:
            response_one = response_df[
                response_df["dataset"] == dataset
            ].copy()

            merged = response_one.merge(
                sig_small,
                on="DepMap_ID",
                how="inner",
                validate="many_to_one",
            )

            for (
                drug_id,
                drug,
            ), drug_df in merged.groupby(
                ["drug_id", "drug"],
                sort=False,
            ):
                data = drug_df[
                    [
                        "DepMap_ID",
                        "tissue",
                        "signature_score",
                        "auc",
                    ]
                ].dropna().copy()

                data = data.drop_duplicates("DepMap_ID")

                tissue_n = (
                    data.groupby("tissue")["DepMap_ID"]
                    .nunique()
                )

                eligible_tissues = tissue_n[
                    tissue_n >= 2
                ].index

                data = data[
                    data["tissue"].isin(eligible_tissues)
                ].copy()

                n_models = data["DepMap_ID"].nunique()
                n_tissues = data["tissue"].nunique()

                if n_models < min_models:
                    rejected["too_few_models"] += 1
                    continue

                if n_tissues < min_tissues:
                    rejected["too_few_tissues"] += 1
                    continue

                data["score_within_tissue"] = (
                    data["signature_score"]
                    - data.groupby("tissue")[
                        "signature_score"
                    ].transform("mean")
                )

                data["auc_within_tissue"] = (
                    data["auc"]
                    - data.groupby("tissue")[
                        "auc"
                    ].transform("mean")
                )

                x_sd = data["score_within_tissue"].std(ddof=0)
                y_sd = data["auc_within_tissue"].std(ddof=0)

                if pd.isna(x_sd) or x_sd <= 0:
                    rejected["zero_score_variance"] += 1
                    continue

                if pd.isna(y_sd) or y_sd <= 0:
                    rejected["zero_auc_variance"] += 1
                    continue

                fit = sm.OLS(
                    data["auc_within_tissue"].astype(float),
                    data[["score_within_tissue"]].astype(float),
                ).fit(cov_type="HC3")

                beta = fit.params["score_within_tissue"]
                beta_std = beta * x_sd / y_sd

                rho, rho_p = spearmanr(
                    data["score_within_tissue"],
                    data["auc_within_tissue"],
                    nan_policy="omit",
                )

                target_values = (
                    drug_df["target"]
                    .dropna()
                    .astype(str)
                )

                target = (
                    target_values.iloc[0]
                    if len(target_values)
                    else pd.NA
                )

                rows.append({
                    "dataset": dataset,
                    "source_cancer": source_cancer,
                    "contrast": contrast,
                    "drug_id": drug_id,
                    "drug": drug,
                    "drug_key": normalize_drug_name(drug),
                    "target": target,
                    "n_models": n_models,
                    "n_tissues": n_tissues,
                    "n_up_genes": sig_small["n_up_used"].iloc[0],
                    "n_down_genes": sig_small["n_down_used"].iloc[0],
                    "beta_tissue_adjusted": beta,
                    "beta_standardized": beta_std,
                    "standard_error_HC3": fit.bse[
                        "score_within_tissue"
                    ],
                    "pvalue": fit.pvalues[
                        "score_within_tissue"
                    ],
                    "spearman_rho_within_tissue": rho,
                    "spearman_pvalue_within_tissue": rho_p,
                    "median_auc": data["auc"].median(),
                })

    results = pd.DataFrame(rows)

    print("Screen rejection counts:", rejected)

    if results.empty:
        
        results = pd.DataFrame(columns=RESULT_COLUMNS)
        raise RuntimeError(
            "No drug-signature tests passed the screening criteria. "
            "Inspect the 'pre_screen_diagnostics.csv' table printed immediately "
            "above. In particular check overlap_models, complete_overlap_models, "
            "eligible_drugs_before_variance_check, and whether the intended "
            "Celligner-aligned expression matrix was loaded."
        )

    results["FDR_within_analysis"] = np.nan

    for _, idx in results.groupby(
        [
            "dataset",
            "source_cancer",
            "contrast",
        ]
    ).groups.items():
        idx = list(idx)

        results.loc[
            idx,
            "FDR_within_analysis",
        ] = bh_fdr(
            results.loc[idx, "pvalue"]
        ).values

    results["FDR_global_all_tests"] = (
        bh_fdr(results["pvalue"]).values
    )

    results["direction"] = np.where(
        results["beta_tissue_adjusted"] < 0,
        "sensitivity",
        "resistance",
    )

    return results


all_results = run_signature_drug_screen(
    responses,
    signature_scores,
)

print("Total drug-signature tests:", len(all_results))

display(
    all_results.groupby(
        [
            "dataset",
            "source_cancer",
            "contrast",
        ]
    ).agg(
        n_tests=("drug_id", "size"),
        n_FDR05=(
            "FDR_within_analysis",
            lambda x: (x < 0.05).sum(),
        ),
        min_FDR=("FDR_within_analysis", "min"),
    )
)

In [ ]:

def save_sorted_results(
    df,
    base_name,
):
    by_fdr = df.sort_values(
        [
            "source_cancer",
            "contrast",
            "FDR_within_analysis",
            "pvalue",
        ],
        ascending=True,
        na_position="last",
    )

    by_p = df.sort_values(
        [
            "source_cancer",
            "contrast",
            "pvalue",
            "FDR_within_analysis",
        ],
        ascending=True,
        na_position="last",
    )

    by_fdr.to_csv(
        OUTPUT_DIR / f"{base_name}_sorted_by_FDR.csv",
        index=False,
    )

    by_p.to_csv(
        OUTPUT_DIR / f"{base_name}_sorted_by_pvalue.csv",
        index=False,
    )

    return by_fdr, by_p


database_sorted = {}

for dataset in [
    "PRISM",
    "GDSC1",
    "GDSC2",
    "CTRP",
]:
    part = all_results[
        all_results["dataset"] == dataset
    ].copy()

    fdr_sorted, p_sorted = save_sorted_results(
        part,
        dataset,
    )

    database_sorted[dataset] = fdr_sorted

    print(
        dataset,
        "rows:",
        len(part),
        "FDR<0.05:",
        (part["FDR_within_analysis"] < 0.05).sum(),
    )


all_by_fdr = all_results.sort_values(
    [
        "source_cancer",
        "contrast",
        "FDR_within_analysis",
        "pvalue",
    ],
    ascending=True,
)

all_by_p = all_results.sort_values(
    [
        "source_cancer",
        "contrast",
        "pvalue",
        "FDR_within_analysis",
    ],
    ascending=True,
)

all_by_fdr.to_csv(
    OUTPUT_DIR / "all_databases_results_sorted_by_FDR.csv",
    index=False,
)

all_by_p.to_csv(
    OUTPUT_DIR / "all_databases_results_sorted_by_pvalue.csv",
    index=False,
)


In [ ]:

for dataset in [
    "PRISM",
    "GDSC1",
    "GDSC2",
    "CTRP",
]:
    print("\n" + "=" * 90)
    print(dataset)
    print("=" * 90)

    for cancer in ["CRC", "UCEC"]:
        for contrast in ["H_vs_L", "H_vs_MSIH"]:
            top = (
                all_results[
                    (all_results["dataset"] == dataset)
                    & (all_results["source_cancer"] == cancer)
                    & (all_results["contrast"] == contrast)
                ]
                .sort_values(
                    [
                        "FDR_within_analysis",
                        "pvalue",
                    ]
                )
                .head(15)
            )

            print(
                f"\n{cancer} / {contrast}"
            )

            display(
                top[
                    [
                        "drug",
                        "target",
                        "n_models",
                        "n_tissues",
                        "beta_tissue_adjusted",
                        "beta_standardized",
                        "pvalue",
                        "FDR_within_analysis",
                        "direction",
                    ]
                ]
            )


In [ ]:

PLATFORMS = [
    "PRISM",
    "GDSC1",
    "GDSC2",
    "CTRP",
]


def choose_target(values):
    values = [
        str(x)
        for x in values
        if pd.notna(x)
        and str(x).strip()
        and str(x).lower() != "nan"
    ]

    return " | ".join(
        dict.fromkeys(values)
    )


consensus_rows = []

for (
    source_cancer,
    contrast,
), block in all_results.groupby(
    ["source_cancer", "contrast"],
    sort=False,
):
    
    block = block.sort_values(
        [
            "FDR_within_analysis",
            "pvalue",
        ]
    ).drop_duplicates(
        [
            "dataset",
            "drug_key",
        ]
    )

    drug_keys = sorted(
        block["drug_key"]
        .dropna()
        .unique()
    )

    for drug_key in drug_keys:
        d = block[
            block["drug_key"] == drug_key
        ].copy()

        row = {
            "source_cancer": source_cancer,
            "contrast": contrast,
            "drug_key": drug_key,
            "drug": d["drug"].iloc[0],
            "target": choose_target(
                d["target"].tolist()
            ),
        }

        signed_zs = []

        for platform in PLATFORMS:
            p = d[
                d["dataset"] == platform
            ]

            if p.empty:
                row[f"beta_{platform}"] = np.nan
                row[f"beta_std_{platform}"] = np.nan
                row[f"pvalue_{platform}"] = np.nan
                row[f"FDR_{platform}"] = np.nan
                row[f"n_{platform}"] = np.nan
                continue

            r = p.iloc[0]

            beta = r["beta_tissue_adjusted"]
            pvalue = r["pvalue"]

            row[f"beta_{platform}"] = beta
            row[f"beta_std_{platform}"] = r[
                "beta_standardized"
            ]
            row[f"pvalue_{platform}"] = pvalue
            row[f"FDR_{platform}"] = r[
                "FDR_within_analysis"
            ]
            row[f"n_{platform}"] = r[
                "n_models"
            ]

            if pd.notna(pvalue) and pd.notna(beta):
                p_clip = float(
                    np.clip(
                        pvalue,
                        1e-300,
                        1.0,
                    )
                )

                z = (
                    np.sign(beta)
                    * norm.isf(p_clip / 2.0)
                )

                signed_zs.append(z)

        beta_cols = [
            f"beta_{p}"
            for p in PLATFORMS
        ]

        betas = np.array(
            [
                row[c]
                for c in beta_cols
                if pd.notna(row[c])
            ],
            dtype=float,
        )

        fdrs = np.array(
            [
                row[f"FDR_{p}"]
                for p in PLATFORMS
                if pd.notna(row[f"FDR_{p}"])
            ],
            dtype=float,
        )

        row["n_platforms"] = len(betas)
        row["n_sensitivity_direction"] = int(
            np.sum(betas < 0)
        )
        row["n_resistance_direction"] = int(
            np.sum(betas > 0)
        )
        row["all_same_direction"] = (
            len(betas) >= 2
            and len(set(np.sign(betas))) == 1
        )
        row["n_FDR05"] = int(
            np.sum(fdrs < 0.05)
        )
        row["n_FDR10"] = int(
            np.sum(fdrs < 0.10)
        )
        row["min_FDR"] = (
            np.min(fdrs)
            if len(fdrs)
            else np.nan
        )

        if len(signed_zs) >= 2:
            meta_z = (
                np.sum(signed_zs)
                / np.sqrt(len(signed_zs))
            )
            meta_p = (
                2 * norm.sf(abs(meta_z))
            )
        else:
            meta_z = np.nan
            meta_p = np.nan

        row["meta_z"] = meta_z
        row["meta_pvalue"] = meta_p

        consensus_rows.append(row)


consensus = pd.DataFrame(
    consensus_rows
)


consensus["meta_FDR"] = np.nan

for _, idx in consensus.groupby(
    ["source_cancer", "contrast"]
).groups.items():
    idx = list(idx)

    consensus.loc[
        idx,
        "meta_FDR",
    ] = bh_fdr(
        consensus.loc[idx, "meta_pvalue"]
    ).values

display(consensus.head())


In [ ]:

def assign_priority(row):
    n = row["n_platforms"]
    neg = row["n_sensitivity_direction"]
    pos = row["n_resistance_direction"]
    sig = row["n_FDR05"]
    meta_fdr = row["meta_FDR"]

    
    
    if (
        n >= 3
        and neg >= 3
        and sig >= 2
        and pd.notna(meta_fdr)
        and meta_fdr < 0.05
    ):
        return "STRONG_SENSITIVITY"

    
    
    if (
        n >= 2
        and neg >= 2
        and sig >= 1
    ):
        return "REPLICATED_SENSITIVITY"

    if (
        n >= 3
        and pos >= 3
        and sig >= 2
        and pd.notna(meta_fdr)
        and meta_fdr < 0.05
    ):
        return "STRONG_RESISTANCE"

    if (
        n >= 2
        and pos >= 2
        and sig >= 1
    ):
        return "REPLICATED_RESISTANCE"

    if (
        n >= 2
        and row["all_same_direction"]
    ):
        return "DIRECTIONALLY_CONSISTENT"

    return "MIXED_OR_WEAK"


priority_order = {
    "STRONG_SENSITIVITY": 0,
    "REPLICATED_SENSITIVITY": 1,
    "DIRECTIONALLY_CONSISTENT": 2,
    "REPLICATED_RESISTANCE": 3,
    "STRONG_RESISTANCE": 4,
    "MIXED_OR_WEAK": 5,
}


consensus["priority"] = consensus.apply(
    assign_priority,
    axis=1,
)

consensus["priority_rank"] = (
    consensus["priority"]
    .map(priority_order)
)

consensus = consensus.sort_values(
    [
        "source_cancer",
        "contrast",
        "priority_rank",
        "meta_FDR",
        "min_FDR",
        "meta_pvalue",
    ],
    ascending=True,
    na_position="last",
).reset_index(drop=True)

consensus.to_csv(
    OUTPUT_DIR / "cross_database_consensus_all.csv",
    index=False,
)

display(
    consensus.groupby(
        [
            "source_cancer",
            "contrast",
            "priority",
        ]
    ).size().rename("n_drugs").reset_index()
)


In [ ]:

for (
    cancer,
    contrast,
), part in consensus.groupby(
    ["source_cancer", "contrast"],
    sort=False,
):
    part = part.sort_values(
        [
            "priority_rank",
            "meta_FDR",
            "min_FDR",
            "meta_pvalue",
        ],
        ascending=True,
        na_position="last",
    )

    filename = (
        f"consensus_{safe_filename(cancer)}_"
        f"{safe_filename(contrast)}.csv"
    )

    part.to_csv(
        OUTPUT_DIR / filename,
        index=False,
    )

    
    part.sort_values(
        [
            "meta_FDR",
            "meta_pvalue",
            "min_FDR",
        ],
        ascending=True,
        na_position="last",
    ).to_csv(
        OUTPUT_DIR
        / (
            f"consensus_{safe_filename(cancer)}_"
            f"{safe_filename(contrast)}_sorted_by_metaFDR.csv"
        ),
        index=False,
    )


In [ ]:

PRIORITY_COLORS = {
    "STRONG_SENSITIVITY": "#7BC67B",
    "REPLICATED_SENSITIVITY": "#C7E9C0",
    "DIRECTIONALLY_CONSISTENT": "#FFF3B0",
    "REPLICATED_RESISTANCE": "#F7C6C7",
    "STRONG_RESISTANCE": "#E88989",
    "MIXED_OR_WEAK": "#EEEEEE",
}


def priority_style(v):
    color = PRIORITY_COLORS.get(
        v,
        "#FFFFFF",
    )
    return (
        f"background-color: {color}; "
        "font-weight: bold;"
    )


def beta_style(v):
    if pd.isna(v):
        return ""

    if v < 0:
        return "background-color: #D9F2D9;"
    if v > 0:
        return "background-color: #F9D6D5;"
    return ""


def fdr_style(v):
    if pd.isna(v):
        return ""

    if v < 0.05:
        return (
            "background-color: #A8DDA8; "
            "font-weight: bold;"
        )

    if v < 0.10:
        return "background-color: #FFF0A6;"

    return ""


def make_colored_consensus(part):
    display_cols = [
        "drug",
        "target",
        "priority",
        "n_platforms",
        "n_sensitivity_direction",
        "n_resistance_direction",
        "n_FDR05",
        "meta_z",
        "meta_pvalue",
        "meta_FDR",
        "min_FDR",
    ]

    for platform in PLATFORMS:
        display_cols.extend(
            [
                f"beta_{platform}",
                f"FDR_{platform}",
            ]
        )

    display_cols = [
        c for c in display_cols
        if c in part.columns
    ]

    view = part[display_cols].copy()

    styler = view.style

    
    
    
    def style_elements(styler, func, subset):
        if hasattr(styler, "map"):
            return styler.map(
                func,
                subset=subset,
            )
        else:
            return styler.applymap(
                func,
                subset=subset,
            )

    
    if "priority" in view.columns:
        styler = style_elements(
            styler,
            priority_style,
            ["priority"],
        )

    
    beta_cols = [
        c for c in view.columns
        if c.startswith("beta_")
        and not c.startswith("beta_std_")
    ]

    if beta_cols:
        styler = style_elements(
            styler,
            beta_style,
            beta_cols,
        )

    
    fdr_cols = [
        c for c in view.columns
        if c.startswith("FDR_")
    ]

    fdr_cols += [
        c for c in [
            "meta_FDR",
            "min_FDR",
        ]
        if c in view.columns
    ]

    if fdr_cols:
        styler = style_elements(
            styler,
            fdr_style,
            fdr_cols,
        )

    
    numeric_format = {}

    for c in view.columns:
        if (
            c.startswith("beta_")
            or c.startswith("FDR_")
            or c in [
                "meta_z",
                "meta_pvalue",
                "meta_FDR",
                "min_FDR",
            ]
        ):
            numeric_format[c] = "{:.4g}"

    styler = (
        styler
        .format(
            numeric_format,
            na_rep="",
        )
        .set_properties(
            **{
                "text-align": "center",
                "font-size": "10pt",
            }
        )
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#2F4858"),
                        ("color", "white"),
                        ("font-weight", "bold"),
                        ("text-align", "center"),
                    ],
                }
            ]
        )
    )

    return styler
for (
    cancer,
    contrast,
), part in consensus.groupby(
    ["source_cancer", "contrast"],
    sort=False,
):
    print("\n" + "=" * 100)
    print(cancer, "/", contrast)
    print("=" * 100)

    top = (
        part.sort_values(
            [
                "priority_rank",
                "meta_FDR",
                "min_FDR",
            ],
            na_position="last",
        )
        .head(100)
    )

    styled = make_colored_consensus(top)
    display(styled)

    html_path = (
        OUTPUT_DIR
        / (
            f"colored_consensus_{safe_filename(cancer)}_"
            f"{safe_filename(contrast)}.html"
        )
    )

    html_path.write_text(
        styled.to_html(),
        encoding="utf-8",
    )

    print("Saved:", html_path)


In [ ]:

sensitivity_candidates = consensus[
    consensus["priority"].isin(
        [
            "STRONG_SENSITIVITY",
            "REPLICATED_SENSITIVITY",
        ]
    )
].copy()

sensitivity_candidates = sensitivity_candidates.sort_values(
    [
        "source_cancer",
        "contrast",
        "priority_rank",
        "meta_FDR",
        "min_FDR",
    ],
    na_position="last",
)

sensitivity_candidates.to_csv(
    OUTPUT_DIR / "replicated_sensitivity_candidates.csv",
    index=False,
)

display(
    sensitivity_candidates[
        [
            "source_cancer",
            "contrast",
            "drug",
            "target",
            "priority",
            "n_platforms",
            "n_sensitivity_direction",
            "n_FDR05",
            "meta_pvalue",
            "meta_FDR",
            "min_FDR",
        ]
    ].head(100)
)


In [ ]:





candidate_small = sensitivity_candidates[
    [
        "source_cancer",
        "contrast",
        "drug_key",
        "drug",
        "target",
        "priority",
        "meta_FDR",
        "min_FDR",
        "n_sensitivity_direction",
        "n_FDR05",
    ]
].copy()

cross_cancer_rows = []

for contrast, block in candidate_small.groupby(
    "contrast"
):
    crc = block[
        block["source_cancer"] == "CRC"
    ].copy()

    ucec = block[
        block["source_cancer"] == "UCEC"
    ].copy()

    merged = crc.merge(
        ucec,
        on="drug_key",
        how="inner",
        suffixes=("_CRC", "_UCEC"),
    )

    if not merged.empty:
        merged["contrast"] = contrast
        cross_cancer_rows.append(merged)


if cross_cancer_rows:
    cross_cancer = pd.concat(
        cross_cancer_rows,
        ignore_index=True,
    )

    cross_cancer = cross_cancer.sort_values(
        [
            "contrast",
            "meta_FDR_CRC",
            "meta_FDR_UCEC",
        ]
    )
else:
    cross_cancer = pd.DataFrame()

cross_cancer.to_csv(
    OUTPUT_DIR / "cross_cancer_replicated_sensitivity_candidates.csv",
    index=False,
)

display(cross_cancer.head(100))


In [ ]:

qc = (
    all_results.groupby(
        [
            "dataset",
            "source_cancer",
            "contrast",
        ]
    )
    .agg(
        n_tests=("drug_id", "size"),
        n_significant_FDR05=(
            "FDR_within_analysis",
            lambda x: (x < 0.05).sum(),
        ),
        n_sensitivity_FDR05=(
            "direction",
            lambda x: (
                (
                    x == "sensitivity"
                )
            ).sum(),
        ),
        min_FDR=("FDR_within_analysis", "min"),
        median_n_models=("n_models", "median"),
        median_n_tissues=("n_tissues", "median"),
    )
    .reset_index()
)


sens_sig_counts = (
    all_results[
        (all_results["FDR_within_analysis"] < 0.05)
        & (all_results["direction"] == "sensitivity")
    ]
    .groupby(
        [
            "dataset",
            "source_cancer",
            "contrast",
        ]
    )
    .size()
    .rename("n_sensitivity_FDR05_correct")
    .reset_index()
)

qc = qc.drop(
    columns=["n_sensitivity_FDR05"]
).merge(
    sens_sig_counts,
    on=[
        "dataset",
        "source_cancer",
        "contrast",
    ],
    how="left",
)

qc[
    "n_sensitivity_FDR05_correct"
] = qc[
    "n_sensitivity_FDR05_correct"
].fillna(0).astype(int)

qc.to_csv(
    OUTPUT_DIR / "analysis_QC_summary.csv",
    index=False,
)

display(qc)


In [ ]:

print("Output directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")
for path in sorted(
    OUTPUT_DIR.iterdir()
):
    print(" -", path.name)
